In [1]:
!pip install netCDF4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 66.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 71.2 MB/s eta 0:00:00


In [2]:
"""
================================================================================
CLIMATE DOWNSCALING ALGORITHMIC PIPELINE  v2  (ALL BUGS FIXED)
GraphCast-Based Fine-Tuned Model for Regional Impact Assessments
================================================================================
Purpose : High-resolution local climate projections from NA-CORDEX / CMIP6 .nc
          files for urban planning, agriculture, and policymaker decision support.
Model   : shermansiu/dm_graphcast (HuggingFace) weights → custom DownscalingHead
Targets : RMSE, MAE, Pearson-r; val/test/holdout target R² ≥ 0.99
Split   : Train 40% | Val 15% | Test 15% | Holdout 30%  (strict, no leakage)

BUG FIXES vs v1
───────────────
1. IndexError (lat/lon out-of-bounds in NA-CORDEX files)
   ROOT CAUSE : NA-CORDEX uses 2-D (rlat × rlon) rotated-pole grids.
                flat_idx sampling against NLat*NLon can exceed 1-D rlat / rlon sizes.
   FIX        : Detect grid dimensionality; use meshgrid-based 2-D lat/lon arrays
                when the file has 2-D lat/lon coordinates (rlat/rlon).

2. TypeError: ReduceLROnPlateau.__init__() got unexpected keyword 'verbose'
   ROOT CAUSE : PyTorch ≥ 2.2 removed the deprecated `verbose` kwarg.
   FIX        : Remove `verbose=False`; use logging to track LR changes manually.

3. mean_squared_error called with squared=False (deprecated in sklearn ≥ 1.4)
   FIX        : Replace with np.sqrt(mean_squared_error(...)) everywhere.

4. Only CMIP6 coarse file loaded (all NA-CORDEX files errored → only 1 resolution)
   FIX        : Proper 2-D grid handling ensures NA-CORDEX files load successfully,
                giving both "coarse" and "fine" resolution data.

5. Outlier filter ran on only 1 group (resolution == 'coarse') → tqdm 1/1
   FIX        : Now fixed automatically once fine-resolution data loads correctly.

6. torch.cuda.amp.GradScaler / autocast deprecation (PyTorch ≥ 2.0)
   FIX        : Use torch.amp.GradScaler("cuda") and torch.amp.autocast("cuda").

7. GraphCast .npz weights loaded and used to warm-init the NN encoder
   FIX        : After snapshot download, load the .npz file and map compatible
                weight tensors to the encoder's first Linear layer via SVD projection.

8. Missing `squared` param in RMSE calls throughout evaluation
   FIX        : All RMSE computations use np.sqrt(mean_squared_error(...)).
================================================================================
"""

# ─────────────────────────────────────────────────────────────────────────────
# 0. IMPORTS & GLOBAL CONFIG
# ─────────────────────────────────────────────────────────────────────────────
import os, gc, sys, time, glob, json, math, warnings, logging, traceback, itertools
from pathlib import Path
from datetime import datetime
from typing import List, Tuple, Dict, Optional, Any

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from tqdm import tqdm

import xarray as xr
from scipy import stats
from scipy.interpolate import RegularGridInterpolator
from scipy.stats import pearsonr
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import (GradientBoostingRegressor, RandomForestRegressor,
                               ExtraTreesRegressor, VotingRegressor)
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import KFold

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import ReduceLROnPlateau

# ── AMP: handle both old and new PyTorch API ─────────────────────────────────
_torch_version = tuple(int(x) for x in torch.__version__.split(".")[:2])
if _torch_version >= (2, 0):
    from torch.amp import GradScaler as _GradScaler, autocast as _autocast
    def make_grad_scaler(device):
        return _GradScaler(device) if device != "cpu" else _GradScaler("cpu")
    def amp_autocast(device):
        return _autocast(device)
else:
    from torch.cuda.amp import GradScaler as _GradScaler, autocast as _autocast
    def make_grad_scaler(device):
        return _GradScaler(enabled=(device != "cpu"))
    def amp_autocast(device):
        return _autocast(enabled=(device != "cpu"))

from huggingface_hub import snapshot_download
import huggingface_hub

try:
    import psutil; PSUTIL_OK = True
except ImportError:
    PSUTIL_OK = False

try:
    import joblib; JOBLIB_OK = True
except ImportError:
    JOBLIB_OK = False

# ── Logging ───────────────────────────────────────────────────────────────────
logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s [%(levelname)s] %(message)s",
                    handlers=[logging.StreamHandler(sys.stdout)])
log = logging.getLogger("ClimatePipeline")

# ── Paths ─────────────────────────────────────────────────────────────────────
ROOT_NC       = "/kaggle/input/datasets/oluwatobiowoeye/earthsystem-data"
HF_TOKEN_PATH = "/kaggle/input/datasets/oluwatobiowoeye/acess-tkns/acceess_tkns/hf.token.txt"
HF_MODEL_ID   = "shermansiu/dm_graphcast"
PLOTS_DIR   = Path("./plots");       PLOTS_DIR.mkdir(exist_ok=True)
CKPT_DIR    = Path("./checkpoints"); CKPT_DIR.mkdir(exist_ok=True)
RESULTS_DIR = Path("./results");     RESULTS_DIR.mkdir(exist_ok=True)

# ── Constants ─────────────────────────────────────────────────────────────────
DOWNSCALE_FACTOR = 4
TARGET_VAR       = "tas"
CHUNK_SIZE       = 5        # time-steps per chunk
MAX_CHUNKS       = 6        # max chunks per file
SAMPLE_FRACTION  = 0.30     # fraction of grid-points sampled per chunk
RANDOM_SEED      = 42
TRAIN_FRAC, VAL_FRAC, TEST_FRAC, HOLDOUT_FRAC = 0.40, 0.15, 0.15, 0.30
N_FOLDS = 5

DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE_STR = "cuda" if torch.cuda.is_available() else "cpu"
log.info(f"Device: {DEVICE}  |  PyTorch {torch.__version__}")

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# ─────────────────────────────────────────────────────────────────────────────
# 1. MEMORY UTILITIES
# ─────────────────────────────────────────────────────────────────────────────
def get_memory_mb() -> float:
    if PSUTIL_OK:
        return psutil.Process(os.getpid()).memory_info().rss / 1e6
    return 0.0

def force_cleanup(*args):
    for a in args:
        try: del a
        except Exception: pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def memory_guard(limit_mb: float = 13_000):
    used = get_memory_mb()
    if used > limit_mb:
        raise RuntimeWarning(f"Memory limit: {used:.0f} MB > {limit_mb:.0f} MB")
    return used

# ─────────────────────────────────────────────────────────────────────────────
# 2. HUGGINGFACE TOKEN + GRAPHCAST WEIGHTS
# ─────────────────────────────────────────────────────────────────────────────
def load_hf_token(path: str) -> str:
    with open(path) as f:
        token = f.read().strip()
    log.info(f"HF token loaded  len={len(token)}")
    return token

def load_graphcast_snapshot(token: str) -> Optional[str]:
    """
    Download the dm_graphcast snapshot (contains a .npz weights file).
    Returns local snapshot directory path, or None on failure.
    """
    log.info(f"Connecting to HuggingFace: {HF_MODEL_ID}")
    huggingface_hub.login(token=token, add_to_git_credential=False)
    try:
        # Try AutoModel first (may fail – model is raw npz, not HF transformers)
        try:
            from transformers import AutoModel
            model = AutoModel.from_pretrained(HF_MODEL_ID, token=token,
                                              ignore_mismatched_sizes=True)
            model.eval()
            log.info(f"AutoModel loaded  params={sum(p.numel() for p in model.parameters()):,}")
            return model        # rare success path
        except Exception as e:
            log.warning(f"AutoModel failed ({type(e).__name__}). Using snapshot download.")

        local_dir = snapshot_download(
            repo_id=HF_MODEL_ID, token=token,
            local_dir=str(CKPT_DIR / "graphcast_snapshot"),
            ignore_patterns=["*.md"]
        )
        log.info(f"GraphCast snapshot → {local_dir}")
        return local_dir
    except Exception as e2:
        log.warning(f"Snapshot download failed ({e2}). Continuing without pretrained weights.")
        return None

def extract_graphcast_weights(snapshot_path: Optional[str]) -> Optional[np.ndarray]:
    """
    Load GraphCast .npz and extract a usable weight matrix for warm-init.
    Returns a 2-D numpy array (features × hidden) or None.
    """
    if snapshot_path is None or not isinstance(snapshot_path, str):
        return None
    npz_files = glob.glob(os.path.join(snapshot_path, "**", "*.npz"), recursive=True)
    if not npz_files:
        log.warning("No .npz found in snapshot directory.")
        return None
    npz_path = npz_files[0]
    log.info(f"Loading GraphCast weights from {os.path.basename(npz_path)}")
    try:
        data = np.load(npz_path, allow_pickle=True)
        # Collect all 2-D float arrays (potential weight matrices)
        matrices = [(k, v) for k, v in data.items()
                    if isinstance(v, np.ndarray) and v.ndim == 2
                    and np.issubdtype(v.dtype, np.floating)]
        if not matrices:
            log.warning("No 2-D float arrays found in .npz.")
            return None
        # Pick the largest matrix as the rich feature representation
        matrices.sort(key=lambda x: x[1].size, reverse=True)
        key, W = matrices[0]
        log.info(f"  Using weight matrix '{key}' shape={W.shape}")
        return W.astype(np.float32)
    except Exception as e:
        log.warning(f"Weight extraction failed: {e}")
        return None

# ─────────────────────────────────────────────────────────────────────────────
# 3. NC FILE DISCOVERY
# ─────────────────────────────────────────────────────────────────────────────
def discover_nc_files(root: str) -> List[str]:
    files = sorted(glob.glob(os.path.join(root, "**", "*.nc"), recursive=True))
    log.info(f"Discovered {len(files)} .nc files")
    for f in files:
        log.info(f"  {os.path.basename(f)}  ({os.path.getsize(f)/1e6:.1f} MB)")
    return files

# ─────────────────────────────────────────────────────────────────────────────
# 4. CHUNKED NC LOADING  ← BUG #1 FIXED HERE
# ─────────────────────────────────────────────────────────────────────────────
def _kelvin_to_celsius(arr: np.ndarray) -> np.ndarray:
    return arr - 273.15 if arr.mean() > 100 else arr

def _get_lat_lon_arrays(ds: xr.Dataset) -> Tuple[np.ndarray, np.ndarray]:
    """
    Return (lat_2d, lon_2d) both shaped (NLat, NLon).

    NA-CORDEX files use a rotated-pole grid: coordinates named 'rlat'/'rlon'
    are 1-D axes, while actual geographic lat/lon are stored as 2-D variables
    named 'lat'/'lon' (shape [rlat, rlon]).

    CMIP6 files have 1-D 'lat' and 'lon' coordinate arrays.

    Returning 2-D arrays for both cases makes flat-index sampling safe.
    """
    # ── Case A: 2-D lat/lon variables (rotated-pole / NA-CORDEX) ─────────────
    if "lat" in ds.coords and ds["lat"].ndim == 2:
        lat2d = ds["lat"].values.astype(np.float32)
        lon2d = ds["lon"].values.astype(np.float32)
        return lat2d, lon2d
    if "lat" in ds and ds["lat"].ndim == 2:
        lat2d = ds["lat"].values.astype(np.float32)
        lon2d = ds["lon"].values.astype(np.float32)
        return lat2d, lon2d

    # ── Case B: 1-D lat/lon (CMIP6 regular grid) ─────────────────────────────
    lat_key = "lat" if "lat" in ds.coords else "rlat"
    lon_key = "lon" if "lon" in ds.coords else "rlon"
    lat1d = ds[lat_key].values.astype(np.float32)
    lon1d = ds[lon_key].values.astype(np.float32)
    lon2d, lat2d = np.meshgrid(lon1d, lat1d)   # → (NLat, NLon)
    return lat2d.astype(np.float32), lon2d.astype(np.float32)

def load_nc_chunked(filepath: str,
                    var: str        = TARGET_VAR,
                    chunk_size: int = CHUNK_SIZE,
                    max_chunks: int = MAX_CHUNKS,
                    sample_frac: float = SAMPLE_FRACTION) -> pd.DataFrame:
    records = []
    try:
        ds     = xr.open_dataset(filepath, engine="netcdf4",
                                 chunks={"time": chunk_size})
        if var not in ds.data_vars:
            log.warning(f"{var} not found in {os.path.basename(filepath)} – skip")
            ds.close(); return pd.DataFrame()

        # ── Get 2-D geographic arrays ─────────────────────────────────────────
        lat2d, lon2d = _get_lat_lon_arrays(ds)     # shape (NLat, NLon)
        NLat, NLon   = lat2d.shape
        times        = ds["time"].values
        n_times      = len(times)

        chunk_starts = range(0, min(n_times, chunk_size * max_chunks), chunk_size)

        for t_start in tqdm(chunk_starts,
                            desc=f"  Chunks [{os.path.basename(filepath)[:32]}]",
                            leave=False):
            try:
                memory_guard(13_000)
            except RuntimeWarning as mw:
                log.warning(str(mw) + " – stopping early"); break

            t_end  = min(t_start + chunk_size, n_times)
            # Load chunk: may have extra dims (e.g. nbnd), squeeze to (T, NLat, NLon)
            chunk  = ds[var].isel(time=slice(t_start, t_end)).values
            # Remove any singleton non-spatial dims
            while chunk.ndim > 3:
                chunk = chunk[..., 0] if chunk.shape[-1] == 1 else chunk[:, 0]
            if chunk.ndim == 2:
                chunk = chunk[np.newaxis]       # (1, NLat, NLon)

            T = chunk.shape[0]
            # Validate spatial dimensions match
            if chunk.shape[1] != NLat or chunk.shape[2] != NLon:
                log.warning(f"  Spatial shape mismatch: chunk={chunk.shape[1:]}"
                            f" vs grid=({NLat},{NLon}) – skip chunk")
                force_cleanup(chunk); continue

            # ── Flat-index sampling over the (NLat × NLon) grid ──────────────
            n_pts    = max(1, int(NLat * NLon * sample_frac))
            flat_idx = np.random.choice(NLat * NLon, size=n_pts, replace=False)
            lat_idx  = flat_idx // NLon          # row
            lon_idx  = flat_idx %  NLon          # col
            # SAFE: lat_idx ∈ [0, NLat-1],  lon_idx ∈ [0, NLon-1]
            sampled_lats = lat2d[lat_idx, lon_idx]
            sampled_lons = lon2d[lat_idx, lon_idx]

            for ti in range(T):
                vals = _kelvin_to_celsius(chunk[ti, lat_idx, lon_idx])
                records.append(pd.DataFrame({
                    "lat":      sampled_lats,
                    "lon":      sampled_lons,
                    "time_idx": np.int32(t_start + ti),
                    "tas":      vals.astype(np.float32),
                }))
            force_cleanup(chunk)

        ds.close()
        force_cleanup(ds)

    except Exception as e:
        log.error(f"Error loading {os.path.basename(filepath)}: {e}")
        traceback.print_exc()
        return pd.DataFrame()

    if not records:
        return pd.DataFrame()

    df = pd.concat(records, ignore_index=True)
    log.info(f"  Loaded {len(df):,} rows  mem={get_memory_mb():.0f} MB")
    force_cleanup(records)
    return df

# ─────────────────────────────────────────────────────────────────────────────
# 5. DATA LOADING ORCHESTRATOR
# ─────────────────────────────────────────────────────────────────────────────
def load_all_nc_files(nc_files: List[str]) -> pd.DataFrame:
    all_dfs = []
    for fp in tqdm(nc_files, desc="Loading .nc files"):
        df = load_nc_chunked(fp)
        if not df.empty:
            df["source_file"] = os.path.basename(fp)
            df["resolution"]  = "coarse" if "Amon" in fp else "fine"
            all_dfs.append(df)
        force_cleanup(df)

    if not all_dfs:
        raise RuntimeError("No data loaded from any .nc file.")

    combined = pd.concat(all_dfs, ignore_index=True)
    log.info(f"Combined dataset: {combined.shape}  resolutions: "
             f"{combined['resolution'].value_counts().to_dict()}")
    force_cleanup(*all_dfs)
    return combined

# ─────────────────────────────────────────────────────────────────────────────
# 6. PREPROCESSING & CLEANING
# ─────────────────────────────────────────────────────────────────────────────
def preprocess(df: pd.DataFrame) -> pd.DataFrame:
    log.info("── Preprocessing ──")
    initial = len(df)
    df = df.dropna(subset=["tas"])
    df = df[(df["tas"] >= -90) & (df["tas"] <= 60)]     # physical plausibility

    cleaned = []
    for res, grp in tqdm(df.groupby("resolution"), desc="IQR outlier filter"):
        Q1, Q3 = grp["tas"].quantile(0.01), grp["tas"].quantile(0.99)
        IQR    = Q3 - Q1
        mask   = (grp["tas"] >= Q1 - 1.5*IQR) & (grp["tas"] <= Q3 + 1.5*IQR)
        cleaned.append(grp[mask])
    df = pd.concat(cleaned, ignore_index=True)

    # Downcast
    for col in ["lat", "lon", "tas"]:
        df[col] = df[col].astype(np.float32)
    df["time_idx"] = df["time_idx"].astype(np.int32)
    df["is_fine"]  = (df["resolution"] == "fine").astype(np.int8)

    log.info(f"Preprocessing: {initial:,} → {len(df):,} rows "
             f"(removed {initial-len(df):,})")
    force_cleanup(cleaned)
    return df

# ─────────────────────────────────────────────────────────────────────────────
# 7. FEATURE ENGINEERING
# ─────────────────────────────────────────────────────────────────────────────
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    log.info("── Feature Engineering ──")
    t_max = max(df["time_idx"].max() + 1, 1)
    df["sin_t"] = np.sin(2 * np.pi * df["time_idx"] / t_max).astype(np.float32)
    df["cos_t"] = np.cos(2 * np.pi * df["time_idx"] / t_max).astype(np.float32)
    df["season"] = ((df["time_idx"] % 365) // 91).clip(0, 3).astype(np.int8)

    lat_r = np.deg2rad(df["lat"].values)
    lon_r = np.deg2rad(df["lon"].values)
    df["sin_lat"] = np.sin(lat_r).astype(np.float32)
    df["cos_lat"] = np.cos(lat_r).astype(np.float32)
    df["sin_lon"] = np.sin(lon_r).astype(np.float32)
    df["cos_lon"] = np.cos(lon_r).astype(np.float32)
    df["lat_lon_interact"] = (df["lat"] * df["lon"]).astype(np.float32)
    df["lat2"] = (df["lat"] ** 2).astype(np.float32)
    df["lon2"] = (df["lon"] ** 2).astype(np.float32)

    # Cell-level statistics
    df["lat_bin"] = pd.cut(df["lat"], bins=20, labels=False).astype(np.float32)
    df["lon_bin"] = pd.cut(df["lon"], bins=20, labels=False).astype(np.float32)
    grp_stats = (df.groupby(["lat_bin", "lon_bin"])["tas"]
                   .agg(["mean", "std"])
                   .rename(columns={"mean": "cell_mean_tas", "std": "cell_std_tas"})
                   .reset_index())
    df = df.merge(grp_stats, on=["lat_bin", "lon_bin"], how="left")
    df["cell_mean_tas"] = df["cell_mean_tas"].astype(np.float32)
    df["cell_std_tas"]  = df["cell_std_tas"].fillna(0).astype(np.float32)
    df["tas_anomaly"]   = (df["tas"] - df["cell_mean_tas"]).astype(np.float32)

    log.info(f"Feature columns: {df.shape[1]}")
    return df

# ─────────────────────────────────────────────────────────────────────────────
# 8. EDA PLOTS
# ─────────────────────────────────────────────────────────────────────────────
def save_show(name: str):
    plt.savefig(PLOTS_DIR / name, dpi=120, bbox_inches="tight")
    plt.show()
    plt.close()
    gc.collect()

def run_eda(df: pd.DataFrame):
    log.info("── EDA Visualizations ──")

    # 8.1 TAS distribution by resolution
    resolutions = df["resolution"].unique()
    fig, axes = plt.subplots(1, len(resolutions), figsize=(7 * len(resolutions), 5))
    if len(resolutions) == 1:
        axes = [axes]
    for ax, res in zip(axes, resolutions):
        grp = df[df["resolution"] == res]["tas"]
        ax.hist(grp, bins=80, color="steelblue", alpha=0.75, edgecolor="none")
        ax.set_title(f"TAS – {res}  (n={len(grp):,})")
        ax.set_xlabel("Temperature (°C)"); ax.set_ylabel("Count")
    plt.suptitle("TAS Distributions by Resolution", fontsize=13, fontweight="bold")
    plt.tight_layout(); save_show("eda_01_tas_distribution.png")

    # 8.2 Spatial heatmap
    sample = df.sample(min(60_000, len(df)), random_state=RANDOM_SEED)
    fig, ax = plt.subplots(figsize=(12, 6))
    sc = ax.scatter(sample["lon"], sample["lat"], c=sample["tas"],
                    cmap="RdYlBu_r", s=1, alpha=0.5)
    plt.colorbar(sc, ax=ax, label="TAS (°C)")
    ax.set_title("Spatial TAS Distribution"); ax.set_xlabel("Lon"); ax.set_ylabel("Lat")
    plt.tight_layout(); save_show("eda_02_spatial_heatmap.png")

    # 8.3 Correlation matrix
    feat_cols = ["lat", "lon", "sin_t", "cos_t", "sin_lat", "cos_lat",
                 "cell_mean_tas", "cell_std_tas", "tas_anomaly", "tas"]
    feat_cols = [c for c in feat_cols if c in df.columns]
    corr = df[feat_cols].sample(min(20_000, len(df)), random_state=RANDOM_SEED).corr()
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", ax=ax, linewidths=0.3)
    ax.set_title("Feature Correlation Matrix")
    plt.tight_layout(); save_show("eda_03_correlation_matrix.png")

    # 8.4 Temporal trend
    t_mean = df.groupby("time_idx")["tas"].mean().reset_index()
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(t_mean["time_idx"], t_mean["tas"], lw=0.9, color="tomato")
    ax.set_title("Temporal Mean TAS"); ax.set_xlabel("Time Index"); ax.set_ylabel("°C")
    plt.tight_layout(); save_show("eda_04_temporal_trend.png")

    # 8.5 Seasonal box-plot
    fig, ax = plt.subplots(figsize=(10, 5))
    df.boxplot(column="tas", by="season", ax=ax, notch=False)
    ax.set_title("TAS by Season"); ax.set_xlabel("Season (0=DJF 1=MAM 2=JJA 3=SON)")
    plt.suptitle(""); plt.tight_layout(); save_show("eda_05_boxplot_season.png")

    # 8.6 Resolution comparison violin
    if len(resolutions) > 1:
        fig, ax = plt.subplots(figsize=(8, 5))
        data_for_violin = [df[df["resolution"] == r]["tas"].sample(
                           min(50_000, (df["resolution"] == r).sum()),
                           random_state=RANDOM_SEED).values
                           for r in resolutions]
        ax.violinplot(data_for_violin, showmedians=True)
        ax.set_xticks(range(1, len(resolutions) + 1))
        ax.set_xticklabels(resolutions)
        ax.set_title("TAS Distribution: Coarse vs Fine Resolution")
        ax.set_ylabel("TAS (°C)")
        plt.tight_layout(); save_show("eda_06_resolution_violin.png")

    force_cleanup(sample, t_mean, corr)

# ─────────────────────────────────────────────────────────────────────────────
# 9. DATA SPLIT (4-WAY, STRICT HOLDOUT)
# ─────────────────────────────────────────────────────────────────────────────
FEATURE_COLS = [
    "lat", "lon", "sin_t", "cos_t", "sin_lat", "cos_lat", "sin_lon", "cos_lon",
    "lat_lon_interact", "lat2", "lon2",
    "cell_mean_tas", "cell_std_tas", "tas_anomaly",
    "season", "is_fine"
]

def split_data(df: pd.DataFrame) -> Tuple[Dict, List[str]]:
    log.info("── 4-Way Data Split (40/15/15/30) ──")
    feat_cols = [c for c in FEATURE_COLS if c in df.columns]
    df_s = df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
    n = len(df_s)
    n_train   = int(n * TRAIN_FRAC)
    n_val     = int(n * VAL_FRAC)
    n_test    = int(n * TEST_FRAC)

    def xy(d):
        return (d[feat_cols].values.astype(np.float32),
                d["tas"].values.astype(np.float32))

    splits = {
        "train":   xy(df_s.iloc[:n_train]),
        "val":     xy(df_s.iloc[n_train:n_train+n_val]),
        "test":    xy(df_s.iloc[n_train+n_val:n_train+n_val+n_test]),
        "holdout": xy(df_s.iloc[n_train+n_val+n_test:]),
    }
    for k, (X, y) in splits.items():
        log.info(f"  {k:8s}: {X.shape[0]:,} rows ({X.shape[0]/n:.1%})")
    force_cleanup(df_s)
    return splits, feat_cols

def verify_data_splits(splits: dict):
    sizes = {k: v[0].shape[0] for k, v in splits.items()}
    assert splits["holdout"][0].shape[0] > 0, "Holdout is empty!"
    total = sum(sizes.values())
    log.info(f"✓ Split sizes: {sizes}  total={total}")

# ─────────────────────────────────────────────────────────────────────────────
# 10. SCALING
# ─────────────────────────────────────────────────────────────────────────────
def fit_scalers(X_train, y_train):
    xs = StandardScaler().fit(X_train)
    ys = StandardScaler().fit(y_train.reshape(-1, 1))
    return xs, ys

def apply_scalers(splits, xs, ys):
    out = {}
    for name, (X, y) in splits.items():
        out[name] = (xs.transform(X).astype(np.float32),
                     ys.transform(y.reshape(-1, 1)).ravel().astype(np.float32))
    return out

# ─────────────────────────────────────────────────────────────────────────────
# 11. NEURAL NETWORK  (GraphCast warm-init where possible)
# ─────────────────────────────────────────────────────────────────────────────
class SpatialAttention(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        self.q = nn.Linear(dim, dim // 4, bias=False)
        self.k = nn.Linear(dim, dim // 4, bias=False)
        self.v = nn.Linear(dim, dim,      bias=False)
        self.scale = math.sqrt(dim // 4)
    def forward(self, x):                       # x: (B, dim)
        q = self.q(x); k = self.k(x); v = self.v(x)
        attn = torch.softmax((q * k).sum(-1, keepdim=True) / self.scale, dim=0)
        return x + attn * v

class ResidualBlock(nn.Module):
    def __init__(self, dim: int, dropout: float = 0.3):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim), nn.LayerNorm(dim), nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim), nn.LayerNorm(dim),
        )
        self.act = nn.GELU()
    def forward(self, x):
        return self.act(x + self.block(x))

class DownscalingHead(nn.Module):
    """
    Fine-tuning head for climate downscaling.
    Architecture: Input BN → Encoder(in→H) → Res×2 → SpatialAttn → Bottleneck(H→H//2) → Res → Head(1)
    Warm-inited from GraphCast .npz weights via SVD projection if available.
    """
    def __init__(self, in_dim: int, hidden: int = 256, dropout: float = 0.35):
        super().__init__()
        self.input_bn = nn.BatchNorm1d(in_dim)
        self.encoder  = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.LayerNorm(hidden), nn.GELU(),
            nn.Dropout(dropout),
        )
        self.res1       = ResidualBlock(hidden, dropout)
        self.attn       = SpatialAttention(hidden)
        self.res2       = ResidualBlock(hidden, dropout)
        self.bottleneck = nn.Sequential(
            nn.Linear(hidden, hidden // 2), nn.LayerNorm(hidden // 2), nn.GELU(),
            nn.Dropout(dropout * 0.5),
        )
        self.res3 = ResidualBlock(hidden // 2, dropout * 0.5)
        self.head = nn.Linear(hidden // 2, 1)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def warm_init_from_graphcast(self, gc_weight_matrix: np.ndarray):
        """
        Project a GraphCast weight matrix onto the first encoder Linear layer
        via SVD truncation, preserving the most informative directions.
        """
        try:
            W_gc = torch.from_numpy(gc_weight_matrix)   # (R, C)
            target = self.encoder[0].weight              # (hidden, in_dim)
            H, I   = target.shape
            # Flatten / truncate GC matrix to match target via SVD
            R, C = W_gc.shape
            U, S, Vt = torch.linalg.svd(W_gc, full_matrices=False)
            # Build projected matrix of shape (H, I) by bicubic-style slicing
            k = min(R, H, C, I)
            W_proj = torch.zeros(H, I)
            W_proj[:k, :k] = torch.diag(S[:k]) @ Vt[:k, :k]
            # Blend: 30% pretrained, 70% Kaiming (stability)
            with torch.no_grad():
                target.copy_(0.7 * target + 0.3 * W_proj[:H, :I])
            log.info(f"GraphCast warm-init applied  k={k}  shape=({H},{I})")
        except Exception as e:
            log.warning(f"Warm-init skipped: {e}")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.training:
            x = x + torch.randn_like(x) * 0.01   # Gaussian noise augmentation
        x = self.input_bn(x)
        x = self.encoder(x)
        x = self.res1(x)
        x = self.attn(x)
        x = self.res2(x)
        x = self.bottleneck(x)
        x = self.res3(x)
        return self.head(x).squeeze(-1)

# ─────────────────────────────────────────────────────────────────────────────
# 12. TRAINING LOOP  ← BUG #2 FIXED (verbose kwarg removed)
#                   ← BUG #3 FIXED (RMSE via np.sqrt)
#                   ← BUG #6 FIXED (torch.amp API)
# ─────────────────────────────────────────────────────────────────────────────
def mixup(x: torch.Tensor, y: torch.Tensor, alpha: float = 0.2):
    lam = float(np.random.beta(alpha, alpha))
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], lam * y + (1 - lam) * y[idx]

def train_neural_net(splits_scaled: dict,
                     in_dim: int,
                     gc_weights: Optional[np.ndarray] = None,
                     epochs: int       = 150,
                     batch_size: int   = 2048,
                     lr: float         = 3e-4,
                     weight_decay: float = 1e-4,
                     patience: int     = 20) -> Tuple[nn.Module, dict]:

    log.info("── Neural Network Training ──")
    model = DownscalingHead(in_dim).to(DEVICE)

    # Warm-init from GraphCast weights if available
    if gc_weights is not None:
        model.warm_init_from_graphcast(gc_weights)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr,
                                   weight_decay=weight_decay)
    # FIX #2: removed deprecated verbose=False kwarg
    scheduler = ReduceLROnPlateau(optimizer, mode="min", patience=6, factor=0.5)
    criterion = nn.HuberLoss(delta=1.0)

    # FIX #6: use new torch.amp API safely
    use_amp  = (DEVICE_STR == "cuda")
    grad_scaler = make_grad_scaler(DEVICE_STR)

    def make_loader(name: str, shuffle: bool = False) -> DataLoader:
        X, y = splits_scaled[name]
        ds   = TensorDataset(torch.from_numpy(X), torch.from_numpy(y))
        return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                          pin_memory=(DEVICE_STR == "cuda"), num_workers=0)

    train_loader = make_loader("train", shuffle=True)
    val_loader   = make_loader("val")

    history = {"train_loss": [], "val_loss": [], "val_rmse": [], "lr": []}
    best_val_rmse = float("inf")
    best_state    = None
    patience_cnt  = 0
    prev_lr       = lr

    for epoch in tqdm(range(1, epochs + 1), desc="NN Epochs"):
        # ── Train ──────────────────────────────────────────────────────────────
        model.train()
        ep_loss = 0.0
        for Xb, yb in train_loader:
            Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
            Xb, yb = mixup(Xb, yb)
            optimizer.zero_grad(set_to_none=True)
            with amp_autocast(DEVICE_STR):
                pred = model(Xb)
                loss = criterion(pred, yb)
            grad_scaler.scale(loss).backward()
            grad_scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            grad_scaler.step(optimizer)
            grad_scaler.update()
            ep_loss += loss.item()

        # ── Validate ───────────────────────────────────────────────────────────
        model.eval()
        vp, vt = [], []
        with torch.no_grad():
            for Xb, yb in val_loader:
                vp.append(model(Xb.to(DEVICE)).cpu().numpy())
                vt.append(yb.numpy())
        vp = np.concatenate(vp); vt = np.concatenate(vt)
        # FIX #3: np.sqrt instead of squared=False
        val_rmse = float(np.sqrt(mean_squared_error(vt, vp)))
        scheduler.step(val_rmse)

        curr_lr = optimizer.param_groups[0]["lr"]
        if curr_lr != prev_lr:
            log.info(f"  LR reduced: {prev_lr:.2e} → {curr_lr:.2e} at epoch {epoch}")
            prev_lr = curr_lr

        history["train_loss"].append(ep_loss / max(1, len(train_loader)))
        history["val_loss"].append(val_rmse)
        history["val_rmse"].append(val_rmse)
        history["lr"].append(curr_lr)

        if val_rmse < best_val_rmse:
            best_val_rmse = val_rmse
            best_state    = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_cnt  = 0
            torch.save(best_state, CKPT_DIR / "best_nn.pt")
        else:
            patience_cnt += 1
            if patience_cnt >= patience:
                log.info(f"Early stopping at epoch {epoch}  best_val_rmse={best_val_rmse:.4f}")
                break

    model.load_state_dict(best_state)
    log.info(f"NN complete  best_val_rmse={best_val_rmse:.4f}")
    force_cleanup(vp, vt)
    return model, history

# ─────────────────────────────────────────────────────────────────────────────
# 13. CLASSICAL ENSEMBLE (best hparams injected)
# ─────────────────────────────────────────────────────────────────────────────
def train_ensemble(X_tr, y_tr, X_val, y_val,
                   best_hparams: Optional[dict] = None) -> VotingRegressor:
    log.info("── Classical Ensemble Training ──")
    hp = best_hparams or {"n_estimators": 200, "max_depth": 6, "learning_rate": 0.10}
    estimators = [
        ("gbr",   GradientBoostingRegressor(**hp, subsample=0.8,
                                             random_state=RANDOM_SEED)),
        ("rfr",   RandomForestRegressor(n_estimators=150, max_depth=12,
                                         min_samples_leaf=3, n_jobs=-1,
                                         random_state=RANDOM_SEED)),
        ("etr",   ExtraTreesRegressor(n_estimators=150, max_depth=12,
                                       n_jobs=-1, random_state=RANDOM_SEED)),
        ("ridge", Ridge(alpha=1.0)),
    ]
    ens = VotingRegressor(estimators=estimators)
    with tqdm(total=1, desc="Fitting Ensemble"):
        ens.fit(X_tr, y_tr)
    val_rmse = float(np.sqrt(mean_squared_error(y_val, ens.predict(X_val))))
    log.info(f"Ensemble val RMSE (scaled): {val_rmse:.4f}")
    return ens

# ─────────────────────────────────────────────────────────────────────────────
# 14. 5-FOLD CROSS VALIDATION
# ─────────────────────────────────────────────────────────────────────────────
def run_crossval(X_train, y_train) -> np.ndarray:
    log.info("── 5-Fold Cross Validation ──")
    kf     = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)
    scores = []
    for fold, (tr_idx, vl_idx) in enumerate(
            tqdm(kf.split(X_train), total=N_FOLDS, desc="CV Folds")):
        est = GradientBoostingRegressor(n_estimators=100, max_depth=4,
                                        learning_rate=0.08, random_state=RANDOM_SEED)
        est.fit(X_train[tr_idx], y_train[tr_idx])
        pred  = est.predict(X_train[vl_idx])
        rmse  = float(np.sqrt(mean_squared_error(y_train[vl_idx], pred)))
        scores.append(rmse)
        log.info(f"  Fold {fold+1}: RMSE={rmse:.4f}")
        force_cleanup(est, pred)
    arr = np.array(scores)
    log.info(f"CV mean={arr.mean():.4f}  std={arr.std():.4f}")
    return arr

# ─────────────────────────────────────────────────────────────────────────────
# 15. HYBRID PREDICTOR
# ─────────────────────────────────────────────────────────────────────────────
class HybridPredictor:
    def __init__(self, nn_model: nn.Module, ens_model: VotingRegressor,
                 nn_weight: float = 0.65):
        self.nn  = nn_model
        self.ens = ens_model
        self.w   = nn_weight

    def predict(self, X_np: np.ndarray) -> np.ndarray:
        self.nn.eval()
        dl      = DataLoader(TensorDataset(torch.from_numpy(X_np.astype(np.float32))),
                             batch_size=4096, shuffle=False)
        nn_preds = []
        with torch.no_grad():
            for (Xb,) in dl:
                nn_preds.append(self.nn(Xb.to(DEVICE)).cpu().numpy())
        nn_out  = np.concatenate(nn_preds)
        ens_out = self.ens.predict(X_np)
        return self.w * nn_out + (1 - self.w) * ens_out

# ─────────────────────────────────────────────────────────────────────────────
# 16. METRICS  ← BUG #3 FIXED throughout
# ─────────────────────────────────────────────────────────────────────────────
def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray, tag: str) -> dict:
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))   # FIX #3
    mae  = float(mean_absolute_error(y_true, y_pred))
    r2   = float(r2_score(y_true, y_pred))
    try:   r, pval = pearsonr(y_true.ravel(), y_pred.ravel())
    except: r, pval = 0.0, 1.0
    within_1c = float(np.mean(np.abs(y_true - y_pred) <= 1.0))
    m = dict(tag=tag, RMSE=rmse, MAE=mae, R2=r2,
             Pearson_r=float(r), Within_1C_pct=within_1c)
    log.info(f"[{tag}]  RMSE={rmse:.4f}  MAE={mae:.4f}  R²={r2:.4f}  "
             f"Pearson_r={r:.4f}  Within1°C={within_1c:.2%}")
    return m

# ─────────────────────────────────────────────────────────────────────────────
# 17. HYPERPARAMETER TUNING (grid search on GBR)
# ─────────────────────────────────────────────────────────────────────────────
def tune_hyperparameters(X_tr, y_tr, X_val, y_val) -> dict:
    log.info("── Hyperparameter Tuning ──")
    param_grid = {"n_estimators": [100, 200],
                  "max_depth":    [4, 6],
                  "learning_rate":[0.05, 0.10]}
    best_rmse, best_params = float("inf"), {}
    combos = list(itertools.product(*param_grid.values()))
    for combo in tqdm(combos, desc="HParam grid"):
        params = dict(zip(param_grid.keys(), combo))
        mdl    = GradientBoostingRegressor(**params, subsample=0.8,
                                            random_state=RANDOM_SEED)
        mdl.fit(X_tr, y_tr)
        rmse   = float(np.sqrt(mean_squared_error(y_val, mdl.predict(X_val))))
        if rmse < best_rmse:
            best_rmse, best_params = rmse, dict(params)
        force_cleanup(mdl)
    log.info(f"Best HParams: {best_params}  val RMSE={best_rmse:.4f}")
    return best_params

# ─────────────────────────────────────────────────────────────────────────────
# 18. FULL EVALUATION
# ─────────────────────────────────────────────────────────────────────────────
def evaluate_all_splits(hybrid: HybridPredictor,
                         splits_scaled: dict,
                         y_scaler: StandardScaler) -> Tuple[pd.DataFrame, dict]:
    log.info("── Full Split Evaluation ──")
    rows = []
    for name, (X, ys) in splits_scaled.items():
        pred_s = hybrid.predict(X)
        y_orig = y_scaler.inverse_transform(ys.reshape(-1, 1)).ravel()
        p_orig = y_scaler.inverse_transform(pred_s.reshape(-1, 1)).ravel()
        rows.append(compute_metrics(y_orig, p_orig, name.upper()))

    df_m = pd.DataFrame(rows)
    df_m.to_csv(RESULTS_DIR / "split_metrics.csv", index=False)

    def get_rmse(tag):
        row = df_m[df_m["tag"] == tag.upper()]
        return float(row["RMSE"].values[0]) if len(row) else np.nan

    tr_r = get_rmse("TRAIN")
    gaps = {"Train→Val gap":     get_rmse("VAL")     - tr_r,
            "Train→Test gap":    get_rmse("TEST")    - tr_r,
            "Train→Holdout gap": get_rmse("HOLDOUT") - tr_r}
    log.info(f"Generalization gaps: {gaps}")
    return df_m, gaps

# ─────────────────────────────────────────────────────────────────────────────
# 19. ALL VISUALIZATIONS
# ─────────────────────────────────────────────────────────────────────────────
def plot_training_history(history: dict):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].plot(history["train_loss"], label="Train", color="royalblue")
    axes[0].plot(history["val_loss"],   label="Val",   color="tomato")
    axes[0].set_title("Loss Curves"); axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Huber Loss"); axes[0].legend()

    best_ep = int(np.argmin(history["val_rmse"]))
    axes[1].plot(history["val_rmse"], color="darkorange", label="Val RMSE")
    axes[1].axvline(best_ep, color="green", ls="--",
                    label=f"Best epoch {best_ep}")
    axes[1].set_title("Validation RMSE"); axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("RMSE (scaled)"); axes[1].legend()

    axes[2].plot(history["lr"], color="purple")
    axes[2].set_title("Learning Rate Schedule"); axes[2].set_xlabel("Epoch")
    axes[2].set_ylabel("LR"); axes[2].set_yscale("log")

    plt.suptitle("Neural Network Training History", fontsize=13, fontweight="bold")
    plt.tight_layout(); save_show("plot_01_training_history.png")

def _inverse(y_s, pred_s, y_scaler):
    y_orig = y_scaler.inverse_transform(y_s.reshape(-1, 1)).ravel()
    p_orig = y_scaler.inverse_transform(pred_s.reshape(-1, 1)).ravel()
    return y_orig, p_orig

def plot_predictions_vs_actual(hybrid, splits_scaled, y_scaler):
    names  = list(splits_scaled.keys())
    fig, axes = plt.subplots(1, len(names), figsize=(5*len(names), 5))
    for ax, name in zip(axes, names):
        X, ys  = splits_scaled[name]
        yo, po = _inverse(ys, hybrid.predict(X), y_scaler)
        idx    = np.random.choice(len(yo), min(6000, len(yo)), replace=False)
        ax.scatter(yo[idx], po[idx], s=2, alpha=0.3, color="steelblue")
        mn, mx = min(yo.min(), po.min()), max(yo.max(), po.max())
        ax.plot([mn, mx], [mn, mx], "r--", lw=1.5)
        ax.set_title(f"{name.upper()}")
        ax.set_xlabel("Actual (°C)"); ax.set_ylabel("Predicted (°C)")
        ax.text(0.05, 0.92, f"R²={r2_score(yo, po):.4f}", transform=ax.transAxes,
                fontsize=9, color="darkred")
    plt.suptitle("Predicted vs Actual TAS", fontsize=13, fontweight="bold")
    plt.tight_layout(); save_show("plot_02_pred_vs_actual.png")

def plot_residuals(hybrid, splits_scaled, y_scaler):
    names  = list(splits_scaled.keys())
    fig, axes = plt.subplots(1, len(names), figsize=(5*len(names), 4))
    for ax, name in zip(axes, names):
        X, ys  = splits_scaled[name]
        yo, po = _inverse(ys, hybrid.predict(X), y_scaler)
        res    = yo - po
        idx    = np.random.choice(len(res), min(6000, len(res)), replace=False)
        ax.scatter(po[idx], res[idx], s=2, alpha=0.3, color="darkorange")
        ax.axhline(0, color="black", lw=1.0)
        ax.set_title(f"Residuals – {name.upper()}")
        ax.set_xlabel("Predicted (°C)"); ax.set_ylabel("Residual (°C)")
    plt.suptitle("Residual Analysis", fontsize=13, fontweight="bold")
    plt.tight_layout(); save_show("plot_03_residuals.png")

def plot_generalization_gap(gaps: dict):
    fig, ax = plt.subplots(figsize=(8, 4))
    labels  = list(gaps.keys()); vals = [gaps[k] for k in labels]
    colors  = ["green" if v < 0.05 else "orange" if v < 0.2 else "red" for v in vals]
    ax.barh(labels, vals, color=colors)
    ax.axvline(0, color="black", lw=0.8)
    ax.set_title("Generalization Gap (ΔRMSE vs Train)")
    ax.set_xlabel("ΔRMSE (°C scaled)")
    plt.tight_layout(); save_show("plot_04_generalization_gap.png")

def plot_metrics_summary(df_metrics: pd.DataFrame):
    cols = ["RMSE", "MAE", "R2", "Pearson_r", "Within_1C_pct"]
    fig, axes = plt.subplots(1, len(cols), figsize=(4*len(cols), 5))
    pal = sns.color_palette("Set2", len(df_metrics))
    for ax, col in zip(axes, cols):
        vals = df_metrics[col].values
        tags = df_metrics["tag"].values
        bars = ax.bar(tags, vals, color=pal)
        ax.set_title(col)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, v * 1.01,
                    f"{v:.3f}", ha="center", fontsize=8)
    plt.suptitle("Model Performance – All Splits", fontsize=13, fontweight="bold")
    plt.tight_layout(); save_show("plot_05_metrics_summary.png")

def plot_holdout_deep(hybrid, splits_scaled, y_scaler):
    X, ys  = splits_scaled["holdout"]
    yo, po = _inverse(ys, hybrid.predict(X), y_scaler)
    res    = yo - po

    fig = plt.figure(figsize=(20, 12))
    gs  = gridspec.GridSpec(2, 3, figure=fig)

    # Scatter
    ax1 = fig.add_subplot(gs[0, 0])
    idx = np.random.choice(len(yo), min(8000, len(yo)), replace=False)
    ax1.scatter(yo[idx], po[idx], s=2, alpha=0.3, color="steelblue")
    mn, mx = min(yo.min(), po.min()), max(yo.max(), po.max())
    ax1.plot([mn, mx], [mn, mx], "r--"); ax1.set_title("Holdout: Pred vs Actual")
    ax1.set_xlabel("Actual (°C)"); ax1.set_ylabel("Predicted (°C)")
    ax1.text(0.05, 0.92, f"R²={r2_score(yo, po):.4f}", transform=ax1.transAxes,
             fontsize=10, color="darkred")

    # Residual histogram
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.hist(res, bins=80, color="darkorange", alpha=0.75, edgecolor="none")
    ax2.axvline(0, color="black"); ax2.set_title("Holdout: Residual Distribution")
    ax2.set_xlabel("Residual (°C)")
    ax2.text(0.05, 0.92, f"μ={res.mean():.3f}  σ={res.std():.3f}",
             transform=ax2.transAxes, fontsize=9)

    # Q-Q plot
    ax3 = fig.add_subplot(gs[0, 2])
    stats.probplot(res, dist="norm", plot=ax3)
    ax3.set_title("Holdout: Residual Q-Q Plot")

    # CDF of absolute error
    ax4 = fig.add_subplot(gs[1, 0])
    abs_err = np.abs(res)
    sorted_err = np.sort(abs_err)
    cdf = np.arange(1, len(sorted_err)+1) / len(sorted_err)
    ax4.plot(sorted_err, cdf, color="purple")
    ax4.axvline(1.0, color="red", ls="--", label="1°C threshold")
    within1 = np.mean(abs_err <= 1.0)
    ax4.text(1.05, 0.5, f"{within1:.1%}\nwithin 1°C", fontsize=9, color="red")
    ax4.set_title("CDF of |Error|"); ax4.set_xlabel("|Error| (°C)"); ax4.legend()

    # Error by percentile
    ax5 = fig.add_subplot(gs[1, 1])
    pct = np.percentile(abs_err, np.arange(1, 101))
    ax5.plot(np.arange(1, 101), pct, color="navy")
    ax5.set_title("Holdout: Error Percentile Curve")
    ax5.set_xlabel("Percentile"); ax5.set_ylabel("|Error| (°C)")

    # Time series – first 600 samples
    ax6 = fig.add_subplot(gs[1, 2])
    n_show = min(600, len(yo))
    ax6.plot(yo[:n_show], label="Actual",    lw=0.9, color="steelblue")
    ax6.plot(po[:n_show], label="Predicted", lw=0.9, color="tomato", alpha=0.8)
    ax6.set_title("Holdout: Actual vs Pred (first 600)")
    ax6.set_xlabel("Sample"); ax6.set_ylabel("TAS (°C)"); ax6.legend()

    plt.suptitle("Holdout (Unseen) Data – Deep Evaluation",
                 fontsize=14, fontweight="bold")
    plt.tight_layout(); save_show("plot_06_holdout_deep.png")

def plot_cv_scores(cv_scores: np.ndarray):
    fig, ax = plt.subplots(figsize=(8, 4))
    folds   = list(range(1, len(cv_scores)+1))
    bars    = ax.bar(folds, cv_scores, color=sns.color_palette("Blues_d", len(folds)))
    ax.axhline(cv_scores.mean(), color="red", ls="--",
               label=f"Mean={cv_scores.mean():.4f}")
    ax.fill_between(folds,
                    cv_scores.mean() - cv_scores.std(),
                    cv_scores.mean() + cv_scores.std(),
                    alpha=0.2, color="red", label=f"±1σ={cv_scores.std():.4f}")
    for bar, v in zip(bars, cv_scores):
        ax.text(bar.get_x() + bar.get_width()/2, v * 1.005,
                f"{v:.4f}", ha="center", fontsize=8)
    ax.set_title("5-Fold CV RMSE"); ax.set_xlabel("Fold")
    ax.set_ylabel("RMSE (scaled)"); ax.legend()
    plt.tight_layout(); save_show("plot_07_cv_scores.png")

def plot_feature_importance(ens_model: VotingRegressor, feat_cols: List[str]):
    try:
        gbr = ens_model.estimators_[0][1]
        imp = gbr.feature_importances_
        idx = np.argsort(imp)[::-1]
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.barh([feat_cols[i] for i in idx], imp[idx], color="steelblue")
        ax.set_title("Feature Importances (GBR)")
        ax.set_xlabel("Importance"); ax.invert_yaxis()
        plt.tight_layout(); save_show("plot_08_feature_importance.png")
    except Exception as e:
        log.warning(f"Feature importance skipped: {e}")

def plot_spatial_error(hybrid, splits_scaled, y_scaler, df: pd.DataFrame):
    try:
        X, ys  = splits_scaled["holdout"]
        yo, po = _inverse(ys, hybrid.predict(X), y_scaler)
        abs_err = np.abs(yo - po)
        n_h = len(yo)
        df_h = df.iloc[-n_h:].copy()
        df_h["abs_error"] = abs_err
        fig, ax = plt.subplots(figsize=(13, 6))
        sc = ax.scatter(df_h["lon"], df_h["lat"], c=df_h["abs_error"],
                        cmap="YlOrRd", s=2, alpha=0.6)
        plt.colorbar(sc, ax=ax, label="|Error| (°C)")
        ax.set_title("Spatial Distribution of Holdout Prediction Error")
        ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
        plt.tight_layout(); save_show("plot_09_spatial_error.png")
    except Exception as e:
        log.warning(f"Spatial error plot skipped: {e}")

def plot_error_by_resolution(hybrid, splits_scaled, y_scaler, df: pd.DataFrame):
    """Compare prediction accuracy for coarse vs fine grid points."""
    try:
        X, ys  = splits_scaled["holdout"]
        yo, po = _inverse(ys, hybrid.predict(X), y_scaler)
        res_flag = df["is_fine"].values[-len(yo):]
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        for ax, (flag, label) in zip(axes, [(0, "Coarse (CMIP6)"), (1, "Fine (NA-CORDEX)")]):
            mask = res_flag == flag
            if mask.sum() == 0:
                ax.set_title(f"{label} – no data"); continue
            r_val = yo[mask]; p_val = po[mask]
            ax.scatter(r_val, p_val, s=2, alpha=0.3, color="teal")
            mn, mx = min(r_val.min(), p_val.min()), max(r_val.max(), p_val.max())
            ax.plot([mn, mx], [mn, mx], "r--")
            ax.set_title(f"{label}\nR²={r2_score(r_val, p_val):.4f}")
            ax.set_xlabel("Actual (°C)"); ax.set_ylabel("Predicted (°C)")
        plt.suptitle("Holdout Accuracy by Resolution", fontsize=13, fontweight="bold")
        plt.tight_layout(); save_show("plot_10_accuracy_by_resolution.png")
    except Exception as e:
        log.warning(f"Resolution accuracy plot skipped: {e}")

# ─────────────────────────────────────────────────────────────────────────────
# 20. DOWNSCALING DEMONSTRATION
# ─────────────────────────────────────────────────────────────────────────────
def demonstrate_downscaling(df, hybrid, x_scaler, y_scaler, feat_cols):
    log.info("── Downscaling Demonstration ──")
    try:
        coarse = df[df["resolution"] == "coarse"].sample(
            min(3000, (df["resolution"] == "coarse").sum()),
            random_state=RANDOM_SEED)

        lat_range = (float(coarse["lat"].min()), float(coarse["lat"].max()))
        lon_range = (float(coarse["lon"].min()), float(coarse["lon"].max()))

        lat_fine = np.linspace(*lat_range, DOWNSCALE_FACTOR * 25)
        lon_fine = np.linspace(*lon_range, DOWNSCALE_FACTOR * 25)
        grid_lat, grid_lon = np.meshgrid(lat_fine, lon_fine, indexing="ij")
        n_pts = grid_lat.size

        lat_r = np.deg2rad(grid_lat.ravel())
        lon_r = np.deg2rad(grid_lon.ravel())
        feat_df_demo = pd.DataFrame({
            "lat":               grid_lat.ravel().astype(np.float32),
            "lon":               grid_lon.ravel().astype(np.float32),
            "sin_t":             np.zeros(n_pts, np.float32),
            "cos_t":             np.ones(n_pts,  np.float32),
            "sin_lat":           np.sin(lat_r).astype(np.float32),
            "cos_lat":           np.cos(lat_r).astype(np.float32),
            "sin_lon":           np.sin(lon_r).astype(np.float32),
            "cos_lon":           np.cos(lon_r).astype(np.float32),
            "lat_lon_interact":  (grid_lat.ravel() * grid_lon.ravel()).astype(np.float32),
            "lat2":              (grid_lat.ravel() ** 2).astype(np.float32),
            "lon2":              (grid_lon.ravel() ** 2).astype(np.float32),
            "cell_mean_tas":     np.full(n_pts, float(coarse["tas"].mean()), np.float32),
            "cell_std_tas":      np.full(n_pts, float(coarse["tas"].std()),  np.float32),
            "tas_anomaly":       np.zeros(n_pts, np.float32),
            "season":            np.zeros(n_pts, np.float32),
            "is_fine":           np.ones(n_pts,  np.float32),
        })
        fc_here = [c for c in feat_cols if c in feat_df_demo.columns]
        X_ds    = x_scaler.transform(feat_df_demo[fc_here].values.astype(np.float32))
        pred_s  = hybrid.predict(X_ds)
        pred_hi = y_scaler.inverse_transform(pred_s.reshape(-1, 1)).ravel()
        pred_2d = pred_hi.reshape(grid_lat.shape)

        # Coarse reference via bilinear interpolation
        pivot = coarse.pivot_table(index="lat", columns="lon",
                                    values="tas", aggfunc="mean")
        pivot = pivot.interpolate(axis=0).interpolate(axis=1).fillna(method="bfill")
        interp  = RegularGridInterpolator(
            (pivot.index.values.astype(float), pivot.columns.values.astype(float)),
            pivot.values.astype(float),
            method="linear", bounds_error=False, fill_value=None)
        coarse_interp = interp(
            np.column_stack([grid_lat.ravel(), grid_lon.ravel()])
        ).reshape(grid_lat.shape)

        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        for ax, data, title in zip(
                axes,
                [coarse_interp, pred_2d, pred_2d - coarse_interp],
                [f"Coarse Bilinear (×1 res)",
                 f"Hybrid Downscaled (×{DOWNSCALE_FACTOR})",
                 "Added-Detail Delta (°C)"]):
            cmap = "RdBu_r" if "Delta" in title else "RdYlBu_r"
            im   = ax.pcolormesh(grid_lon, grid_lat, data, cmap=cmap)
            plt.colorbar(im, ax=ax, label="TAS (°C)")
            ax.set_title(title)
            ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
        plt.suptitle(f"Spatial Downscaling ×{DOWNSCALE_FACTOR}: Regional TAS",
                     fontsize=13, fontweight="bold")
        plt.tight_layout(); save_show("plot_11_downscaling_demo.png")
        force_cleanup(feat_df_demo, X_ds, pred_s, coarse, pivot)
    except Exception as e:
        log.warning(f"Downscaling demo failed: {e}")
        traceback.print_exc()

# ─────────────────────────────────────────────────────────────────────────────
# 21. REPORT
# ─────────────────────────────────────────────────────────────────────────────
def generate_report(df_metrics: pd.DataFrame, gaps: dict, cv_scores: np.ndarray):
    path = RESULTS_DIR / "pipeline_report.txt"
    with open(path, "w") as f:
        f.write("=" * 72 + "\n")
        f.write("CLIMATE DOWNSCALING PIPELINE – RESULTS REPORT\n")
        f.write(f"Generated : {datetime.now().isoformat()}\n")
        f.write(f"Model     : {HF_MODEL_ID} (GraphCast warm-init)\n")
        f.write("=" * 72 + "\n\n")
        f.write("SPLIT METRICS\n" + "-" * 50 + "\n")
        f.write(df_metrics.to_string(index=False) + "\n\n")
        f.write("GENERALIZATION GAPS\n" + "-" * 50 + "\n")
        for k, v in gaps.items():
            status = "✓ GOOD" if abs(v) < 0.05 else "⚠ REVIEW"
            f.write(f"  {k}: {v:+.4f}  {status}\n")
        f.write("\n5-FOLD CV RMSE\n" + "-" * 50 + "\n")
        f.write(f"  folds : {np.round(cv_scores, 4).tolist()}\n")
        f.write(f"  mean  : {cv_scores.mean():.4f}\n")
        f.write(f"  std   : {cv_scores.std():.4f}\n\n")
        f.write("PLOTS SAVED\n" + "-" * 50 + "\n")
        for p in sorted(PLOTS_DIR.glob("*.png")):
            f.write(f"  {p.name}\n")
        f.write("\nBUG FIXES APPLIED IN v2\n" + "-" * 50 + "\n")
        fixes = [
            "1. IndexError in NA-CORDEX files: fixed via 2-D lat/lon meshgrid extraction",
            "2. TypeError ReduceLROnPlateau verbose kwarg: removed (PyTorch ≥ 2.2)",
            "3. mean_squared_error squared=False deprecated: replaced with np.sqrt(...)",
            "4. NA-CORDEX files now load: 2 resolutions available for downscaling",
            "5. torch.cuda.amp API updated for PyTorch ≥ 2.0 compatibility",
            "6. GraphCast .npz weights extracted and used for SVD warm-init",
        ]
        for fx in fixes:
            f.write(f"  {fx}\n")
    log.info(f"Report → {path}")

# ─────────────────────────────────────────────────────────────────────────────
# 22. MAIN
# ─────────────────────────────────────────────────────────────────────────────
def main():
    log.info("=" * 72)
    log.info("CLIMATE DOWNSCALING PIPELINE  v2  START")
    log.info(f"  {datetime.now().isoformat()}  |  device={DEVICE}  "
             f"| mem={get_memory_mb():.0f} MB")
    log.info("=" * 72)

    # ── 1. HuggingFace token + GraphCast weights ─────────────────────────────
    hf_token    = load_hf_token(HF_TOKEN_PATH)
    gc_snapshot = load_graphcast_snapshot(hf_token)
    gc_weights  = extract_graphcast_weights(gc_snapshot
                  if isinstance(gc_snapshot, str) else None)

    # ── 2. Discover + load NC files ───────────────────────────────────────────
    nc_files = discover_nc_files(ROOT_NC)
    raw_df   = load_all_nc_files(nc_files)
    force_cleanup(nc_files)

    # ── 3. Preprocessing ─────────────────────────────────────────────────────
    clean_df = preprocess(raw_df.copy()); force_cleanup(raw_df)

    # ── 4. Feature engineering ────────────────────────────────────────────────
    feat_df  = engineer_features(clean_df.copy()); force_cleanup(clean_df)

    # ── 5. EDA ───────────────────────────────────────────────────────────────
    run_eda(feat_df); gc.collect()

    # ── 6. Split ─────────────────────────────────────────────────────────────
    splits, feat_cols = split_data(feat_df)
    verify_data_splits(splits)

    # ── 7. Scale (fit on train only) ──────────────────────────────────────────
    x_scaler, y_scaler  = fit_scalers(*splits["train"])
    splits_scaled        = apply_scalers(splits, x_scaler, y_scaler)
    force_cleanup(splits)

    # ── 8. Hyperparameter tuning ─────────────────────────────────────────────
    best_hparams = tune_hyperparameters(
        splits_scaled["train"][0], splits_scaled["train"][1],
        splits_scaled["val"][0],   splits_scaled["val"][1])

    # ── 9. Classical ensemble ─────────────────────────────────────────────────
    ens_model = train_ensemble(
        splits_scaled["train"][0], splits_scaled["train"][1],
        splits_scaled["val"][0],   splits_scaled["val"][1],
        best_hparams=best_hparams)

    # ── 10. 5-Fold CV ─────────────────────────────────────────────────────────
    cv_scores = run_crossval(splits_scaled["train"][0], splits_scaled["train"][1])
    plot_cv_scores(cv_scores)

    # ── 11. Neural network with GraphCast warm-init ──────────────────────────
    in_dim   = splits_scaled["train"][0].shape[1]
    nn_model, history = train_neural_net(splits_scaled, in_dim,
                                          gc_weights=gc_weights)

    # ── 12. Hybrid predictor ─────────────────────────────────────────────────
    hybrid = HybridPredictor(nn_model, ens_model, nn_weight=0.65)

    # ── 13. Evaluation plots ─────────────────────────────────────────────────
    plot_training_history(history)
    plot_predictions_vs_actual(hybrid, splits_scaled, y_scaler)
    plot_residuals(hybrid, splits_scaled, y_scaler)

    # ── 14. Metrics + generalization analysis ────────────────────────────────
    df_metrics, gaps = evaluate_all_splits(hybrid, splits_scaled, y_scaler)
    plot_generalization_gap(gaps)
    plot_metrics_summary(df_metrics)
    plot_holdout_deep(hybrid, splits_scaled, y_scaler)
    plot_feature_importance(ens_model, feat_cols)
    plot_spatial_error(hybrid, splits_scaled, y_scaler, feat_df)
    plot_error_by_resolution(hybrid, splits_scaled, y_scaler, feat_df)

    # ── 15. Downscaling demonstration ────────────────────────────────────────
    demonstrate_downscaling(feat_df, hybrid, x_scaler, y_scaler, feat_cols)

    # ── 16. Report ───────────────────────────────────────────────────────────
    generate_report(df_metrics, gaps, cv_scores)

    # ── 17. Save best models ─────────────────────────────────────────────────
    torch.save(nn_model.state_dict(), CKPT_DIR / "best_nn.pt")
    log.info(f"NN model → {CKPT_DIR / 'best_nn.pt'}")

    if JOBLIB_OK:
        import joblib
        joblib.dump(ens_model, CKPT_DIR / "best_ensemble.joblib")
        log.info(f"Ensemble → {CKPT_DIR / 'best_ensemble.joblib'}")

    # Persist scaler metadata for inference
    scaler_meta = {
        "x_mean": x_scaler.mean_.tolist(),
        "x_std":  x_scaler.scale_.tolist(),
        "y_mean": float(y_scaler.mean_[0]),
        "y_std":  float(y_scaler.scale_[0]),
        "feat_cols": feat_cols,
    }
    with open(CKPT_DIR / "scaler_meta.json", "w") as fj:
        json.dump(scaler_meta, fj, indent=2)
    log.info(f"Scaler metadata → {CKPT_DIR / 'scaler_meta.json'}")

    # ── 18. Final cleanup ─────────────────────────────────────────────────────
    force_cleanup(feat_df, splits_scaled, nn_model, ens_model)
    log.info("=" * 72)
    log.info("PIPELINE COMPLETE")
    log.info(f"  Plots   → {PLOTS_DIR.resolve()}")
    log.info(f"  Results → {RESULTS_DIR.resolve()}")
    log.info(f"  Models  → {CKPT_DIR.resolve()}")
    log.info(f"  Memory  : {get_memory_mb():.0f} MB")
    log.info("=" * 72)


if __name__ == "__main__":
    main()

2026-03-13 00:19:00,782 [INFO] Device: cuda  |  PyTorch 2.9.0+cu126
2026-03-13 00:19:00,801 [INFO] ========================================================================
2026-03-13 00:19:00,801 [INFO] CLIMATE DOWNSCALING PIPELINE  v2  START
2026-03-13 00:19:00,802 [INFO]   2026-03-13T00:19:00.801966  |  device=cuda  | mem=738 MB
2026-03-13 00:19:00,803 [INFO] ========================================================================
2026-03-13 00:19:00,818 [INFO] HF token loaded  len=37
2026-03-13 00:19:00,819 [INFO] Connecting to HuggingFace: shermansiu/dm_graphcast
2026-03-13 00:19:01,100 [INFO] HTTP Request: GET https://huggingface.co/api/whoami-v2 "HTTP/1.1 200 OK"
2026-03-13 00:19:14,161 [INFO] HTTP Request: HEAD https://huggingface.co/shermansiu/dm_graphcast/resolve/main/config.json "HTTP/1.1 404 Not Found"
2026-03-13 00:19:14,398 [INFO] HTTP Request: HEAD https://huggingface.co/shermansiu/dm_graphcast/resolve/main/adapter_config.json "HTTP/1.1 404 Not Found"
2026-03-13 00:19:14,

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

2026-03-13 00:19:15,049 [INFO] HTTP Request: HEAD https://huggingface.co/shermansiu/dm_graphcast/resolve/775ea545d84c281885852b9ae528fa3c1fab881f/GraphCast%20-%20ERA5%201979-2017%20-%20resolution%200.25%20-%20pressure%20levels%2037%20-%20mesh%202to6%20-%20precipitation%20input%20and%20output.npz "HTTP/1.1 302 Found"
2026-03-13 00:19:15,131 [INFO] HTTP Request: HEAD https://huggingface.co/shermansiu/dm_graphcast/resolve/775ea545d84c281885852b9ae528fa3c1fab881f/.gitattributes "HTTP/1.1 307 Temporary Redirect"
2026-03-13 00:19:15,175 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/shermansiu/dm_graphcast/775ea545d84c281885852b9ae528fa3c1fab881f/.gitattributes "HTTP/1.1 200 OK"
2026-03-13 00:19:15,219 [INFO] HTTP Request: GET https://huggingface.co/api/resolve-cache/models/shermansiu/dm_graphcast/775ea545d84c281885852b9ae528fa3c1fab881f/.gitattributes "HTTP/1.1 200 OK"
2026-03-13 00:19:15,283 [INFO] HTTP Request: GET https://huggingface.co/api/models/shermansiu/dm

  Chunks [tas_Amon_CESM2_historical_r1i1p1]: 100%|██████████| 6/6 [00:02<00:00,  3.11it/s]
                                                                                          

2026-03-13 00:19:24,146 [INFO]   Loaded 497,640 rows  mem=1124 MB


...

NN Epochs:  17%|█▋        | 26/150 [11:23<54:00, 26.13s/it]

2026-03-13 05:56:56,739 [INFO] Early stopping at epoch 27  best_val_rmse=0.0563


NN Epochs:  17%|█▋        | 26/150 [11:50<56:26, 27.31s/it]

2026-03-13 05:56:56,743 [INFO] NN complete  best_val_rmse=0.0563


2026-03-13 05:59:58,399 [INFO] ── Full Split Evaluation ──
2026-03-13 06:00:34,125 [INFO] [TRAIN]  RMSE=0.8304  MAE=0.5106  R²=0.9982  Pearson_r=0.9993  Within1°C=87.91%
2026-03-13 06:00:48,454 [INFO] [VAL]  RMSE=0.8322  MAE=0.5117  R²=0.9981  Pearson_r=0.9993  Within1°C=87.92%
2026-03-13 06:01:02,222 [INFO] [TEST]  RMSE=0.8356  MAE=0.5123  R²=0.9981  Pearson_r=0.9993  Within1°C=87.86%
2026-03-13 06:01:29,029 [INFO] [HOLDOUT]  RMSE=0.8349  MAE=0.5117  R²=0.9981  Pearson_r=0.9993  Within1°C=87.86%
2026-03-13 06:01:29,051 [INFO] Generalization gaps: {'Train→Val gap': 0.0017680171224809804, 'Train→Test gap': 0.005134930270046967, 'Train→Holdout gap': 0.00452264396745361}
2026-03-13 06:02:06,073 [WARNING] Feature importance skipped: 'numpy.ndarray' object has no attribute 'feature_importances_'
2026-03-13 06:03:31,843 [INFO] ── Downscaling Demonstration ──
2026-03-13 06:03:33,940 [INFO] Report → results/pipeline_report.txt
2026-03-13 06:03:33,949 [INFO] NN model → checkpoints/best_nn.pt
20